## Project 2 Code Review

Our project compares the diets of two different islands in Hawai'i: Oahu and Moloka'i. Oahu's role as the biggest tourist island in Hawai'i as well as the most urbanized has resulted in a rapid change to the diet as it quickly shifted towards hyper-processed western convenience foods.
On the other hand, Moloka'i is a smaller island that has mostly retained the traditional Hawaiian diet -- fresh, raw, healthier foods.

In [5]:
import pandas as pd
import numpy as np
import requests

In [12]:
# turn sheet into df
columns = [
    "Nutrition", "Source",
    "C_1_3",
    "F_4_8", "M_4_8",
    "F_9_13", "M_9_13",
    "F_14_18", "M_14_18",
    "F_19_30", "M_19_30",
    "F_31_50", "M_31_50",
    "F_51_plus", "M_51_plus"
]

data = [
    ["Energy","---",1000,1200,1400,1600,1800,1800,2200,2000,2400,1800,2200,1600,2000],
    ["Protein","RDA",13,19,19,34,34,46,52,46,56,46,56,46,56],
    ["Fiber","---",14,16.8,19.6,22.4,25.2,25.2,30.8,28,33.6,25.2,30.8,22.4,28],
    ["Folate","RDA",150,200,200,300,300,400,400,400,400,400,400,400,400],
    ["Calcium","RDA",700,1000,1000,1300,1300,1300,1300,1000,1000,1000,1000,1200,1000],
    ["Carbohydrate","RDA",130,130,130,130,130,130,130,130,130,130,130,130,130],
    ["Iron","RDA",7,10,10,8,8,15,11,18,8,18,8,8,8],
    ["Magnesium","RDA",80,130,130,240,240,360,410,310,400,320,420,320,420],
    ["Niacin","RDA",6,8,8,12,12,14,16,14,16,14,16,14,16],
    ["Phosphorus","RDA",460,500,500,1250,1250,1250,1250,700,700,700,700,700,700],
    ["Potassium","AI",3000,3800,3800,4500,4500,4700,4700,4700,4700,4700,4700,4700,4700],
    ["Riboflavin","RDA",0.5,0.6,0.6,0.9,0.9,1,1.3,1.1,1.3,1.1,1.3,1.1,1.3],
    ["Thiamin","RDA",0.5,0.6,0.6,0.9,0.9,1,1.2,1.1,1.2,1.1,1.2,1.1,1.2],
    ["Vitamin_A","RDA",300,400,400,600,600,700,900,700,900,700,900,700,900],
    ["Vitamin_B12","RDA",0.9,1.2,1.2,1.8,1.8,2.4,2.4,2.4,2.4,2.4,2.4,2.4,2.4],
    ["Vitamin_B6","RDA",0.5,0.6,0.6,1,1,1.2,1.3,1.3,1.3,1.3,1.3,1.5,1.7],
    ["Vitamin_C","RDA",15,25,25,45,45,65,75,75,90,75,90,75,90],
    ["Vitamin_E","RDA",6,7,7,11,11,15,15,15,15,15,15,15,15],
    ["Vitamin_K","AI",30,55,55,60,60,75,75,90,120,90,120,90,120],
    ["Zinc","RDA",3,5,5,8,8,9,11,8,11,8,11,8,11]
]

# need to map names to fix later issues
dri_name_map = {
    "Fiber": "Fiber, total dietary",
    "Folate": "Folate, DFE",
    "Carbohydrate": "Carbohydrate, by difference",
    "Iron": "Iron, Fe",
    "Magnesium": "Magnesium, Mg",
    "Phosphorus": "Phosphorus, P",
    "Potassium": "Potassium, K",
    "Vitamin_A": "Vitamin A, RAE",
    "Vitamin_B12": "Vitamin B-12",
    "Vitamin_B6": "Vitamin B-6",
    "Vitamin_C": "Vitamin C, total ascorbic acid",
    "Vitamin_E": "Vitamin E (alpha-tocopherol)",
    "Vitamin_K": "Vitamin K (phylloquinone)",
    "Zinc": "Zinc, Zn",
    "Calcium": "Calcium, Ca"
}

dri_df = pd.DataFrame(data, columns=columns)

dri_df["Nutrition"] = dri_df["Nutrition"].replace(dri_name_map)

In [13]:
# need to map age and sex to correct columns for the DRI function
def age_sex_column(age, sex):
    
    if age <= 3:
        return "C_1_3"
    elif age <= 8:
        return f"{sex}_4_8"
    elif age <= 13:
        return f"{sex}_9_13"
    elif age <= 18:
        return f"{sex}_14_18"
    elif age <= 30:
        return f"{sex}_19_30"
    elif age <= 50:
        return f"{sex}_31_50"
    else:
        return f"{sex}_51_plus"

In [14]:
def get_dri(age, sex):
    col = age_sex_column(age, sex)
    
    if col not in dri_df.columns:
        raise ValueError("Invalid age/sex combination.")
    
    result = dri_df.set_index("Nutrition")[col]
    
    return result

In [15]:
# Dietary Reference Intake (DRI) function
# Use "M" or "F" for male or female respectively
get_dri(17, "M")

Nutrition
Energy                            2200.0
Protein                             52.0
Fiber, total dietary                30.8
Folate, DFE                        400.0
Calcium, Ca                       1300.0
Carbohydrate, by difference        130.0
Iron, Fe                            11.0
Magnesium, Mg                      410.0
Niacin                              16.0
Phosphorus, P                     1250.0
Potassium, K                      4700.0
Riboflavin                           1.3
Thiamin                              1.2
Vitamin A, RAE                     900.0
Vitamin B-12                         2.4
Vitamin B-6                          1.3
Vitamin C, total ascorbic acid      75.0
Vitamin E (alpha-tocopherol)        15.0
Vitamin K (phylloquinone)           75.0
Zinc, Zn                            11.0
Name: M_14_18, dtype: float64

In [18]:
foods_to_search = [
    #  Diet Moloka'i
    169308,	#Taro, raw (kalo)
    170431,	#Poi (fermented taro paste)
    2346404,	#Sweet potato, raw (ʻuala)
    171714,	#Breadfruit, raw (ʻulu)
    2747673,	#Fish, ahi/tuna raw
    171959,	#Fish, mahi-mahi (dolphinfish), raw
    173676,	#Fish, opah (moonfish), raw
    170090,	#Seaweed, agar, dried
    170169,	#Coconut meat, raw
    173944,	#Banana, raw 
    169926,	#Papaya, raw
    171477,	#Chicken, whole, raw (kalua-style)
    2195577,	#Pork, shoulder, raw (kalua pork base)
    169303,	#Sweet potato leaves, raw

    # Diet Oahu
    1927473,	#Spam, classic (canned pork)
    168878,	#White rice, cooked (calrose)
    169839,	#Macaroni salad (restaurant-style)
    2706930,	#Hamburger patty, fast food (Big Mac-style)
    170698,	#French fries, fast food
    2706093,	#Chicken nuggets, fast food
    2708615,	#Pizza, cheese, frozen/fast food
    174852,	#Soda, cola (Coca-Cola)
    353617,	#Bread, white sandwich loaf
    748967,	#Eggs, large, raw (whole)
    746782,	#Milk, whole, fluid
    173261,	#Breakfast cereal, corn flakes
    172946,	#Hot dog / frankfurter
    167575,	#Ice cream, vanilla
    167455	#Potato chips, plain
]

In [31]:
apikey = "uOgTaIqGwW8Of1b6UcucKDg1yWcBt01FPea0hqDT"

def get_food_data(fdc_id):
    url = f"https://api.nal.usda.gov/fdc/v1/food/{fdc_id}"
    params = {"api_key": apikey}
    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("Error retrieving food:", fdc_id)
        return None

    food = response.json()

    row = {
        "fdcId": fdc_id,
        "Food": food.get("description", "Unknown")
    }

    for nutrient in food["foodNutrients"]:

        nid = nutrient["nutrient"]["id"]

        if nid in important_nutrients:
            name = important_nutrients[nid]
            row[name] = nutrient.get("amount", 0)

    return row

In [32]:
important_nutrients = {
    1008: "Energy",
    1003: "Protein",
    1005: "Carbohydrate, by difference",
    1079: "Fiber, total dietary",
    1087: "Calcium, Ca",
    1089: "Iron, Fe",
    1092: "Potassium, K",
    1162: "Vitamin C, total ascorbic acid",
    1090: "Magnesium, Mg",
    1095: "Zinc, Zn",
    1091: "Phosphorus, P",
    1165: "Thiamin",
    1166: "Riboflavin",
    1167: "Niacin",
    1177: "Folate, DFE",
    1106: "Vitamin A, RAE",
    1178: "Vitamin B-12",
    1175: "Vitamin B-6",
    1109: "Vitamin E (alpha-tocopherol)",
    1185: "Vitamin K (phylloquinone)",
    1093: "Sodium, Na"
}


def extract_nutrients(food_item):
    nutrients = {v: 0 for v in important_nutrients.values()}
    
    for n in food_item["foodNutrients"]:
        if n["nutrientId"] in important_nutrients:
            nutrients[important_nutrients[n["nutrientId"]]] = n["value"]
    
    return nutrients

In [35]:
rows = []

for fdc in foods_to_search:
    row = get_food_data(fdc)

    if row is not None:
        rows.append(row)

food_df = pd.DataFrame(rows)
food_df = food_df.set_index("Food")
food_df = food_df.fillna(0)

food_df.to_csv("hawaii_food_nutrients.csv")
food_df = pd.read_csv("hawaii_food_nutrients.csv", index_col="Food")

food_df

Error retrieving food: 748967
Error retrieving food: 746782
Error retrieving food: 167455


,fdcId,Energy,Protein,"Carbohydrate, by difference","Fiber, total dietary","Calcium, Ca","Iron, Fe","Magnesium, Mg","Phosphorus, P","Potassium, K",...,"Vitamin C, total ascorbic acid",Thiamin,Riboflavin,Niacin,Vitamin B-6,"Folate, DFE",Vitamin B-12,"Vitamin A, RAE",Vitamin E (alpha-tocopherol),Vitamin K (phylloquinone)
Food,,,,,,,,,,,,,,,,,,,,,
"Taro, raw",169308,112.0,1.500000,26.460000,4.1,43.000,0.5500,33.00,84.00,591.0,...,4.50,0.095,0.025,0.6000,0.2830,22.0,0.000,4.0,2.38,1.0
Poi,170431,112.0,0.380000,27.230000,0.4,16.000,0.8800,24.00,39.00,183.0,...,4.00,0.130,0.040,1.1000,0.2730,21.0,0.000,3.0,2.30,1.0
"Sweet potatoes, orange flesh, without skin, raw",2346404,0.0,1.578125,17.327875,0.0,22.330,0.3980,19.14,36.73,486.4,...,14.84,0.045,0.000,0.4325,0.1241,0.0,0.000,0.0,0.00,0.2
"Breadfruit, raw",171714,103.0,1.070000,27.120000,4.9,17.000,0.5400,25.00,30.00,490.0,...,29.00,0.110,0.030,0.9000,0.1000,14.0,0.000,0.0,0.10,0.5
"Tuna, ahi or yellowfin, frozen, wild caught",2747673,0.0,24.700000,-0.104500,0.0,3.193,0.5914,35.48,270.80,420.2,...,0.00,0.000,0.000,0.0000,0.0000,0.0,1.374,0.0,0.00,0.0
"Fish, mahimahi, raw",171959,85.0,18.500000,0.000000,0.0,15.000,1.1300,30.00,143.00,416.0,...,0.00,0.020,0.070,6.1000,0.4000,5.0,0.600,54.0,0.00,0.0
"Fish, monkfish, raw",173676,76.0,14.480000,0.000000,0.0,8.000,0.3200,21.00,200.00,400.0,...,1.00,0.025,0.060,2.1000,0.2400,7.0,0.900,12.0,0.00,0.0
"Seaweed, agar, dried",170090,306.0,6.210000,80.880000,7.7,625.000,21.4000,770.00,52.00,1125.0,...,0.00,0.010,0.222,0.2020,0.3030,580.0,0.000,0.0,5.00,24.4
"Nuts, coconut meat, raw",170169,354.0,3.330000,15.230000,9.0,14.000,2.4300,32.00,113.00,356.0,...,3.30,0.066,0.020,0.5400,0.0540,26.0,0.000,0.0,0.24,0.2


In [54]:
# add prices to food_df and categorize molokai and oahu
# prices are per 100g

molokai_prices = {
    169308: 0.55,
    170431: 1.32,
    2346404: 0.42,
    171714: 0.77,
    2747673: 3.30,
    171959: 2.86,
    173676: 3.75,
    170090: 15.86,
    170169: 0.57,
    173944: 0.20,
    169926: 0.33,
    171477: 0.55,
    2195577: 0.77,
    169303: 1.10
}

oahu_prices = {
    1927473: 1.26,
    168878: 0.33,
    169839: 1.10,
    2706930: 2.50,
    170698: 2.98,
    2706093: 3.70,
    2708615: 2.00,
    174852: 0.35,
    353617: 0.88,
    748967: 0.92,
    746782: 0.20,
    173261: 1.30,
    172946: 0.94,
    167575: 0.79,
    167455: 2.20
}

food_df["Prices"] = food_df["fdcId"].map(
    lambda x: molokai_prices.get(x, oahu_prices.get(x)))

food_df["Diet_Type"] = food_df["fdcId"].map(
    lambda x: "Molokai" if x in molokai_prices else "Oahu")

food_df

,fdcId,Energy,Protein,"Carbohydrate, by difference","Fiber, total dietary","Calcium, Ca","Iron, Fe","Magnesium, Mg","Phosphorus, P","Potassium, K",...,Niacin,Vitamin B-6,"Folate, DFE",Vitamin B-12,"Vitamin A, RAE",Vitamin E (alpha-tocopherol),Vitamin K (phylloquinone),Diet_Type,Price_2024,Prices
Food,,,,,,,,,,,,,,,,,,,,,
"Taro, raw",169308,112.0,1.500000,26.460000,4.1,43.000,0.5500,33.00,84.00,591.0,...,0.6000,0.2830,22.0,0.000,4.0,2.38,1.0,Molokai,0.55,0.55
Poi,170431,112.0,0.380000,27.230000,0.4,16.000,0.8800,24.00,39.00,183.0,...,1.1000,0.2730,21.0,0.000,3.0,2.30,1.0,Molokai,1.32,1.32
"Sweet potatoes, orange flesh, without skin, raw",2346404,0.0,1.578125,17.327875,0.0,22.330,0.3980,19.14,36.73,486.4,...,0.4325,0.1241,0.0,0.000,0.0,0.00,0.2,Molokai,0.42,0.42
"Breadfruit, raw",171714,103.0,1.070000,27.120000,4.9,17.000,0.5400,25.00,30.00,490.0,...,0.9000,0.1000,14.0,0.000,0.0,0.10,0.5,Molokai,0.77,0.77
"Tuna, ahi or yellowfin, frozen, wild caught",2747673,0.0,24.700000,-0.104500,0.0,3.193,0.5914,35.48,270.80,420.2,...,0.0000,0.0000,0.0,1.374,0.0,0.00,0.0,Molokai,3.30,3.30
"Fish, mahimahi, raw",171959,85.0,18.500000,0.000000,0.0,15.000,1.1300,30.00,143.00,416.0,...,6.1000,0.4000,5.0,0.600,54.0,0.00,0.0,Molokai,2.86,2.86
"Fish, monkfish, raw",173676,76.0,14.480000,0.000000,0.0,8.000,0.3200,21.00,200.00,400.0,...,2.1000,0.2400,7.0,0.900,12.0,0.00,0.0,Molokai,3.75,3.75
"Seaweed, agar, dried",170090,306.0,6.210000,80.880000,7.7,625.000,21.4000,770.00,52.00,1125.0,...,0.2020,0.3030,580.0,0.000,0.0,5.00,24.4,Molokai,15.86,15.86
"Nuts, coconut meat, raw",170169,354.0,3.330000,15.230000,9.0,14.000,2.4300,32.00,113.00,356.0,...,0.5400,0.0540,26.0,0.000,0.0,0.24,0.2,Molokai,0.57,0.57


### Function to Solve Subsistence Problem

In [53]:
from scipy.optimize import linprog

class DietResult:
    def __init__(self, cost, diet, nutrition):
        self.Cost = cost
        self.Diet = diet
        self.Nutrition = nutrition

    def __repr__(self):
        s = f"Cost of diet: ${self.Cost:.2f} per day.\n\n"
        s += "Diet (in 100g units):\n"
        s += self.Diet.sort_values(ascending=False).to_string()
        s += "\n\nNutritional outcomes:\n"
        s += self.Nutrition.to_string()
        return s


def subsistence_solver(food_df, dri_df, age_sex_col, diet_type="All"):

    # Filter diet type
    if diet_type != "All":
        food_df = food_df[food_df["Diet_Type"].str.contains(diet_type)].copy()

    # Nutrient columns in dataset
    nutrient_cols = [
        col for col in food_df.columns
        if col not in ["Prices","fdcId","Diet_Type"]]

    # Map DRI names to food_df names
    nutrient_name_map = {
        "Energy":"Energy",
        "Protein":"Protein",
        "Fiber":"Fiber, total dietary",
        "Folate":"Folate, DFE",
        "Calcium":"Calcium, Ca",
        "Carbohydrate":"Carbohydrate, by difference",
        "Iron":"Iron, Fe",
        "Magnesium":"Magnesium, Mg",
        "Niacin":"Niacin",
        "Phosphorus":"Phosphorus, P",
        "Potassium":"Potassium, K",
        "Riboflavin":"Riboflavin",
        "Thiamin":"Thiamin",
        "Vitamin_A":"Vitamin A, RAE",
        "Vitamin_B12":"Vitamin B-12",
        "Vitamin_B6":"Vitamin B-6",
        "Vitamin_C":"Vitamin C, total ascorbic acid",
        "Vitamin_E":"Vitamin E (alpha-tocopherol)",
        "Vitamin_K":"Vitamin K (phylloquinone)",
        "Zinc":"Zinc, Zn",
        "Sodium":"Sodium, Na"
    }

    # Pull requirements from DRI table
    lower_bounds = {}
    for nut in nutrient_cols:

        dri_name = None
        for k,v in nutrient_name_map.items():
            if v == nut:
                dri_name = k

        if dri_name is not None and dri_name in dri_df["Nutrition"].values:
            val = dri_df.loc[dri_df["Nutrition"]==dri_name, age_sex_col].values[0]
        else:
            val = 0

        lower_bounds[nut] = val


    # minimize cost
    c = food_df["Prices"].values

    # Nutrient matrix
    A = np.array([food_df[n].values for n in nutrient_cols])
    b = np.array([lower_bounds[n] for n in nutrient_cols])

    # Solver
    res = linprog(
        c,
        A_ub = -A,
        b_ub = -b,
        bounds = [(0,None)]*len(c),
        method = "highs")

    if not res.success:
        raise ValueError(res.message)

    # Diet quantities
    diet_quantities = pd.Series(res.x, index=food_df.index)

    # Cost
    total_cost = (diet_quantities * food_df["Prices"]).sum()

    # Nutrition outcomes
    nutrient_totals = pd.Series(A @ res.x, index=nutrient_cols)
    rec_values = pd.Series(lower_bounds)

    # Constraints
    tol = 1e-3
    binding = abs(nutrient_totals - rec_values) < tol

    nutrition_df = pd.DataFrame({
        "Outcome": nutrient_totals,
        "Recommendation": rec_values,
        "Binding": binding.replace({True:"Binding",False:""})})

    return DietResult(
        total_cost,
        diet_quantities[diet_quantities>1e-4],
        nutrition_df)

In [51]:
# Molokai diet for M_19_30
subsistence_solver(food_df, dri_df, "M_19_30", diet_type = "Molokai")

Cost of diet: $7.43 per day.

Diet (in 100g units):
Food
Bananas, raw                                                       17.385092
Taro, raw                                                           6.303047
Chicken, broilers or fryers, breast, meat only, cooked, roasted     0.889609

Nutritional outcomes:
                                     Outcome  Recommendation  Binding
Energy                           2400.000000          2400.0  Binding
Protein                            56.000000            56.0  Binding
Carbohydrate, by difference       563.854129             0.0         
Fiber, total dietary               71.043733             0.0         
Calcium, Ca                       371.300639             0.0         
Iron, Fe                            8.911994             0.0         
Magnesium, Mg                     703.196709             0.0         
Phosphorus, P                    1114.758914             0.0         
Potassium, K                    10176.703839             0.

In [52]:
# Oahu diet for M_19_30
subsistence_solver(food_df, dri_df, "M_19_30", diet_type = "Oahu")

Cost of diet: $7.91 per day.

Diet (in 100g units):
Food
Rice, white, long-grain, regular, enriched, cooked    10.458932
Ice creams, vanilla                                    4.850141
Oscar Mayer, Ham (chopped with natural juice)          0.668097

Nutritional outcomes:
                                    Outcome  Recommendation  Binding
Energy                          2483.897819          2400.0         
Protein                           56.000000            56.0  Binding
Carbohydrate, by difference      411.530000             0.0         
Fiber, total dietary               7.578672             0.0         
Calcium, Ca                      731.420263             0.0         
Iron, Fe                          13.855757             0.0         
Magnesium, Mg                    208.775388             0.0         
Phosphorus, P                   1105.312112             0.0         
Potassium, K                    1504.945899             0.0         
Sodium, Na                      1232.9